In [33]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OrdinalEncoder

import seaborn as sns
df = sns.load_dataset('titanic')
df = df[['sex', 'class', 'age', 'fare', 'embarked', 'survived']].dropna()
df['embarked']

0      S
1      C
2      S
3      S
4      S
      ..
885    Q
886    S
887    S
889    C
890    Q
Name: embarked, Length: 712, dtype: object

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 712 entries, 0 to 890
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   sex       712 non-null    object  
 1   class     712 non-null    category
 2   age       712 non-null    float64 
 3   fare      712 non-null    float64 
 4   embarked  712 non-null    object  
 5   survived  712 non-null    int64   
dtypes: category(1), float64(2), int64(1), object(2)
memory usage: 34.2+ KB


In [35]:
df = df[['survived', 'age', 'class', 'sex', 'embarked']].dropna()
train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)

X = train_df[['age', 'class', 'sex', 'embarked']]
y = train_df['survived']

In [36]:
class_priors = y.value_counts(normalize=True)

num_stats = train_df.groupby('survived')['age'].agg(['mean', 'var'])

cat_cols = ['class', 'sex', 'embarked']
cat_matrices = {col: pd.crosstab(train_df[col], train_df['survived'], normalize='columns') for col in cat_cols}

In [37]:
survived_chance = 1
unsirvived_chance = 1
for col in cat_cols:
    counts = train_df[col].value_counts()
    target_counts = train_df.groupby(col)['survived'].value_counts(normalize=True).unstack()
    for cat_val in counts.index: 
        target_1 = sum((train_df[col] == cat_val) & (train_df['survived'] == 1))
        target_0 = sum((train_df[col] == cat_val) & (train_df['survived'] == 0))
        chance_1 = target_1 / sum(train_df['survived'] == 1)
        chance_0 = target_0 / sum(train_df['survived'] == 0)
        survived_chance *= chance_1
        unsirvived_chance *= chance_0 
        print(f"признак {cat_val}: {target_1} / {target_0} ({chance_1:.2f} / {chance_0:.2f})")

if survived_chance > unsirvived_chance:
    print(f"{survived_chance:.15f}, {unsirvived_chance:.15f}, Survived")
else:
    print(f"{survived_chance:.15f}, {unsirvived_chance:.15f}, Not Survived")
        

признак Third: 67 / 218 (0.29 / 0.64)
признак First: 90 / 53 (0.39 / 0.16)
признак Second: 72 / 70 (0.31 / 0.21)
признак male: 68 / 286 (0.30 / 0.84)
признак female: 161 / 55 (0.70 / 0.16)
признак S: 157 / 282 (0.69 / 0.83)
признак C: 66 / 39 (0.29 / 0.11)
признак Q: 6 / 20 (0.03 / 0.06)
0.000039074693213, 0.000015306214842, Survived


C:\Users\augoa\AppData\Local\Temp\ipykernel_16540\1471515528.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  target_counts = train_df.groupby(col)['survived'].value_counts(normalize=True).unstack()


In [38]:
total_1 = sum(train_df['survived'] == 1)
total_0 = sum(train_df['survived'] == 0)

prior_1 = total_1 / len(train_df)
prior_0 = total_0 / len(train_df)

probs = {1: {}, 0: {}}

for col in cat_cols:
    probs[1][col] = {}
    probs[0][col] = {}
    
    for val in train_df[col].unique():
        target_1 = sum((train_df[col] == val) & (train_df['survived'] == 1))
        target_0 = sum((train_df[col] == val) & (train_df['survived'] == 0))
        
        probs[1][col][val] = target_1 / total_1
        probs[0][col][val] = target_0 / total_0
        print(f"признак {col}={val}: {target_1} / {target_0} ({probs[1][col][val]:.2f} / {probs[0][col][val]:.2f})")
        

признак class=First: 90 / 53 (0.39 / 0.16)
признак class=Second: 72 / 70 (0.31 / 0.21)
признак class=Third: 67 / 218 (0.29 / 0.64)
признак sex=female: 161 / 55 (0.70 / 0.16)
признак sex=male: 68 / 286 (0.30 / 0.84)
признак embarked=C: 66 / 39 (0.29 / 0.11)
признак embarked=S: 157 / 282 (0.69 / 0.83)
признак embarked=Q: 6 / 20 (0.03 / 0.06)


In [39]:
predictions = []

for _, row in test_df.iterrows():
    survived_chance = prior_1
    unsurvived_chance = prior_0
    
    for col in cat_cols:
        val = row[col]
        
        chance_1 = probs[1][col].get(val, 1e-6)
        chance_0 = probs[0][col].get(val, 1e-6)
        
        survived_chance *= chance_1
        unsurvived_chance *= chance_0
        
    pred = 1 if survived_chance > unsurvived_chance else 0
    predictions.append(pred)

accuracy = (predictions == test_df['survived']).mean()
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 73.94%


In [40]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [42]:
import numpy as np

from my_ml_vectorized_models import NaiveBayesVectorized
X_train_np = np.array(X_train, dtype=float)
y_train_np = np.array(y_train)

X_test_np = np.array(X_test, dtype=float)
y_test_np = np.array(y_test)
my_model = NaiveBayesVectorized()
my_model.fit(X_train_np, y_train_np)
predictions = my_model.predict(X_test_np)

ValueError: could not convert string to float: 'Second'